In [ ]:
import torch
import os
import sys
import numpy as np
import pandas as pd
from tabulate import tabulate

# --- 1. 设置路径与导入 ---
# 将项目根目录加入路径，确保能导入 src
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.append(project_root)

# 导入你的实现
from src.Ego2ExoFollowShot.dimension.metrics import calculate_all_flow_metrics
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
from utils.video_kit import load_video_to_gpu

# 导入 VBench (假设已 pip install vbench)
try:
    import vbench
    print(f"✅ VBench version: {vbench.__version__}")
except ImportError:
    print("❌ VBench not found. Please run `pip install vbench`")
    exit()

# --- 2. 配置测试参数 ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VIDEO_PATH = "assets/test_video.mp4" # 替换为你本地的一个真实测试视频路径
GT_PATH = "assets/test_gt.mp4"       # 如果要测 OFC，需要 GT 视频

# 确保视频存在
if not os.path.exists(VIDEO_PATH):
    print(f"⚠️ Video not found at {VIDEO_PATH}, generating dummy video...")
    # 生成一个假视频用于测试 (如果没有真实视频)
    import cv2
    writer = cv2.VideoWriter(VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 30, (224, 224))
    for i in range(30):
        frame = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        writer.write(frame)
    writer.release()
    # 复制一份作为 GT
    import shutil
    shutil.copy(VIDEO_PATH, GT_PATH)

# --- 3. 运行 VBench (基准) ---
print("\n🚀 Running VBench (Official)...")
# VBench 的调用方式可能随版本变化，以下是通用用法
# 通常 VBench 接受视频路径列表
from vbench import VBench
# 初始化 VBench，指定需要计算的维度
my_vbench = VBench(device=DEVICE, full_info_dir="./vbench_info", output_path="./vbench_out")

# 注意：VBench 的 API 有时比较重，这里假设我们调用其底层函数或通过 standard 接口
# 为了演示，我们模拟调用 specific metric class，或者你可以直接使用 my_vbench.evaluate
# 这里我们假设使用 VBench 的 evaluate 接口
vbench_results = {}
try:
    # 这里的 dimension_list 需要根据 VBench 的实际命名调整
    # 例如 'temporal_flickering', 'motion_smoothness', 'dynamic_degree'
    vb_scores = my_vbench.evaluate(
        videos_path=VIDEO_PATH,
        task_list=["temporal_flickering", "motion_smoothness", "dynamic_degree"]
    )
    # 假设返回的是 dataframe 或 dict
    vbench_results = vb_scores.to_dict('records')[0] # 取第一行
except Exception as e:
    print(f"VBench run failed: {e}")
    print("Trying explicit class usage (fallback)...")
    # 如果 VBench 封装太深，这里你可以手动 import 它的具体类
    # from vbench.temporal_flickering import TemporalFlickering
    # ...

# --- 4. 运行你的实现 (Ours) ---
print("\n🚀 Running My Implementation...")

# 加载视频到 GPU Tensor
tensor_gen = load_video_to_gpu(VIDEO_PATH, device=DEVICE, target_size=(224, 224))
tensor_gt = load_video_to_gpu(GT_PATH, device=DEVICE, target_size=(224, 224))

# 预加载模型 (模拟 Bench 的 prepare)
flow_model = raft_small(weights=Raft_Small_Weights.DEFAULT).to(DEVICE).eval()

# 计算
my_results = calculate_all_flow_metrics(
    gen_frames=tensor_gen,
    gt_frames=tensor_gt,
    metrics_to_compute={'tf', 'ms', 'dd', 'ofc'},
    flow_model=flow_model,
    device=DEVICE
)

# --- 5. 对比结果 ---
print("\n📊 Comparison Table:")

comparison_data = []
metrics_map = {
    'TF (Flickering)': ('temporal_flickering', 'tf'),
    'MS (Smoothness)': ('motion_smoothness', 'ms'),
    'DD (Dynamic)':    ('dynamic_degree', 'dd'),
}

for label, (vb_key, my_key) in metrics_map.items():
    vb_val = vbench_results.get(vb_key, -1.0) # -1 if not found
    my_val = my_results.get(my_key, -1.0)
    
    # 计算差异百分比
    diff = abs(vb_val - my_val)
    if vb_val != 0:
        diff_pct = (diff / vb_val) * 100
    else:
        diff_pct = 0.0 if diff == 0 else float('inf')
    
    comparison_data.append([label, f"{vb_val:.4f}", f"{my_val:.4f}", f"{diff_pct:.2f}%"])

print(tabulate(comparison_data, headers=["Metric", "VBench Value", "My Value", "Diff %"], tablefmt="grid"))

# --- 6. 结果分析建议 ---
print("\n🔍 Analysis:")
print("1. 如果 Diff < 5%: 完美对齐。")
print("2. 如果 Diff 在 10%-20%: 正常误差。")
print("   可能原因：")
print("   - Resize 算法不同 (Bilinear vs Bicubic)")
print("   - 归一化方式不同 (0-1 vs -1-1 vs 0-255)")
print("   - 光流模型不同 (RAFT-small vs RAFT-large vs PWC-Net)")
print("3. 如果 Diff 数量级不同 (e.g. 0.1 vs 100):")
print("   - 检查是否需要乘以或除以 255/100 等缩放因子。")